In [13]:
import logging
from turtle import pd
import sys, os
sys.path.append("../..")
from src.pipelines.raw.kinexon import (
    get_detected_events_for_fixture as kinexon_get_detected_events_for_fixture,
    get_positions_for_session as kinexon_get_positions_for_session,
    get_sessions_for_team as kinexon_get_sessions_for_team,
    get_teams_for_season as kinexon_get_teams_for_season,
)
from src.pipelines.raw.sportradar import (
    get_competition_id as sr_get_competition_id,
    get_season_id as sr_get_season_id,
    get_teams_for_season as sr_get_teams_for_season,
    get_fixtures_for_season as sr_get_fixtures_for_season,
    get_fixture_events as sr_get_fixture_events,
    get_players_for_fixture as sr_get_players_for_fixture,
)
from src.pipelines.normalized.match_detected_shots import normalize_match_detected_shots
from src.pipelines.normalized.match_positions import normalize_match_positions
from src.pipelines.normalized.match_events import normalize_match_events
from src.pipelines.normalized.match_events import normalize_match_events_goals, normalize_match_events_setup
from src.pipelines.normalized.match_players import normalize_match_players

from src.pipelines.synced.players import extract_players_for_match
from src.pipelines.synced.shot_events import sync_shot_events, render_shot_event

from src.pipelines.normalized.matches import normalize_matches

from src.pipelines.features.calc_xg_features import calculate_xg_features

from dotenv import load_dotenv
from src.hbl_etl_dagster.utils.api_helper import (
    get_api_kinexon,
    get_api_sportradar,
)
import pandas as pd

In [2]:
# api init
api_sportradar = get_api_sportradar()
api_kinexon = get_api_kinexon()

YEAR = 2025
COMPETITION_NAME = "1. Handball-Bundesliga"

POSITIONS_CACHE_PATH = "../../data/positions/"


logging.basicConfig(level=logging.WARNING)

In [3]:
# Fetch competition and season IDs
competition_id = sr_get_competition_id(api_sportradar, COMPETITION_NAME)
season_id = sr_get_season_id(api_sportradar, competition_id, YEAR)
print(f"Using competition_id: {competition_id} for {COMPETITION_NAME},\nseason_id: {season_id} for year {YEAR}")

Using competition_id: 4c445e5c-3956-11ef-9d0e-b74f5c057367 for 1. Handball-Bundesliga,
season_id: b23932f0-5ca5-11f0-a95f-55240abd96d4 for year 2025


In [4]:
# Fetch teams and fixtures for the season
teams_sportradar = sr_get_teams_for_season(api_sportradar, season_id)
display(teams_sportradar.head(18))

,entity_id,name_full_local,name_full_latin,code_local,code_latin,external_id
0,fea4237d-3952-11ef-9fcd-af5c55c3771d,FRISCH AUF! Göppingen,FRISCH AUF! Göppingen,FAG,FAG,11
1,0045fbc4-3953-11ef-a217-af5c55c3771d,Handball Sport Verein Hamburg,Handball Sport Verein Hamburg,HSV,HSV,127
2,fe9e91c7-3952-11ef-bd62-af5c55c3771d,GWD Minden,GWD Minden,GWD,GWD,10
3,febf038e-3952-11ef-b7c2-af5c55c3771d,THW Kiel,THW Kiel,THW,THW,18
4,fe8d1885-3952-11ef-9130-af5c55c3771d,HC Erlangen,HC Erlangen,HCE,HCE,6
5,febb3114-3952-11ef-b6f7-af5c55c3771d,TVB Stuttgart,TVB Stuttgart,TVB,TVB,17
6,fe911367-3952-11ef-9131-af5c55c3771d,VfL Gummersbach,VfL Gummersbach,GUM,GUM,7
7,feb41b46-3952-11ef-9df6-af5c55c3771d,MT Melsungen,MT Melsungen,MTM,MTM,15
8,fe848316-3952-11ef-8185-af5c55c3771d,SC Magdeburg,SC Magdeburg,SCM,SCM,4
9,fef51771-3952-11ef-97b4-af5c55c3771d,ThSV Eisenach,ThSV Eisenach,EIS,EIS,32


In [5]:
fixtures = sr_get_fixtures_for_season(api_sportradar, season_id)
display(fixtures)

,fixtureId,organizationId,organization,seasonId,season,practiceDrillType,internationalReference,status,fixtureNumber,nameLocal,...,profileId,includeInStandings,updated,added,estimatedFinishTimeUTC,seriesFixtureNumber,discipline,broadcasts,sellout,liveDataAvailable
0,0000798a-5ca7-11f0-a2fe-df9b3a075d9a,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,SCHEDULED,None,TSV Hannover-Burgdorf vs MT Melsungen,...,None,True,2025-07-17T07:44:50,2025-07-09T09:27:57,2026-06-07T01:00:00,None,None,[],NaN,NaN
1,0062111b-5ca7-11f0-93a5-d509415b4f76,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,SCHEDULED,None,Füchse Berlin vs SG Flensburg-Handewitt,...,None,True,2025-07-17T07:44:44,2025-07-09T09:27:58,2026-06-07T01:00:00,None,None,[],NaN,NaN
2,00ba9627-5ca6-11f0-ac5e-5389986df98b,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,MT Melsungen vs. SC DHfK Leipzig,...,None,True,2025-12-31T04:31:59,2025-07-09T09:20:49,2025-10-10T20:00:00,None,None,[],False,NaN
3,00ec13a6-5ca6-11f0-a5f8-51a43a77a48d,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,TSV Hannover-Burgdorf vs. ThSV Eisenach,...,None,True,2025-12-31T04:32:02,2025-07-09T09:20:49,2025-10-11T19:00:00,None,None,[],False,NaN
4,0172aa0a-5ca6-11f0-be58-55240abd96d4,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,SG Flensburg-Handewitt vs. THW Kiel,...,None,True,2025-12-31T04:32:02,2025-07-09T09:20:50,2025-10-11T16:40:00,None,None,[],False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,fd21f3c1-5ca5-11f0-acb1-51a43a77a48d,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,GWD Minden vs. TBV Lemgo Lippe,...,None,True,2025-12-31T04:31:56,2025-07-09T09:20:43,2025-10-04T20:00:00,None,None,[],False,NaN
302,fdad7170-5ca6-11f0-8c4f-7129937f02a0,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,SCHEDULED,None,HC Erlangen vs Bergischer HC,...,None,True,2025-07-17T07:44:57,2025-07-09T09:27:53,2026-06-03T01:00:00,None,None,[],NaN,NaN
303,fe512125-5ca5-11f0-b38f-6b4ccd828704,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,HC Erlangen vs. Füchse Berlin,...,None,True,2025-12-31T04:31:57,2025-07-09T09:20:45,2025-10-04T21:00:00,None,None,[],False,NaN
304,febf38b7-5ca5-11f0-afe0-51a43a77a48d,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,TVB Stuttgart vs. MT Melsungen,...,None,True,2025-12-31T04:31:58,2025-07-09T09:20:46,2025-10-05T19:00:00,None,None,[],False,NaN


In [6]:
# keep only those fixtures that are not in the future
from datetime import datetime
print(f"Total fixtures fetched: {len(fixtures)}")
fixtures = fixtures[fixtures["startTimeLocal"] <= datetime.now().isoformat()]
print(f"Fixtures until now: {len(fixtures)}")
# sort by startTimeLocal
fixtures = fixtures.sort_values(by="startTimeLocal").reset_index(drop=True)
# rename fixtureId to fixture_id for consistency
fixtures = fixtures.rename(columns={"fixtureId": "fixture_id"})
display(fixtures)

Total fixtures fetched: 306
Fixtures until now: 172


,fixture_id,organizationId,organization,seasonId,season,practiceDrillType,internationalReference,status,fixtureNumber,nameLocal,...,profileId,includeInStandings,updated,added,estimatedFinishTimeUTC,seriesFixtureNumber,discipline,broadcasts,sellout,liveDataAvailable
0,ccb6fb61-5ca5-11f0-b602-991bd49151a6,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,TSV Hannover-Burgdorf vs. VfL Gummersbach,...,None,True,2025-12-31T04:31:11,2025-07-09T09:19:22,2025-08-27T20:00:00,None,None,[],False,False
1,d7e307a9-5ca5-11f0-af4b-018ba42b97a2,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,HC Erlangen vs. THW Kiel,...,None,True,2025-12-31T04:31:12,2025-07-09T09:19:40,2025-08-28T20:00:00,None,None,[],False,NaN
2,c9ebe4b9-5ca5-11f0-b9d0-d509415b4f76,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,MT Melsungen vs. Rhein-Neckar Löwen,...,None,True,2025-12-31T04:31:10,2025-07-09T09:19:17,2025-08-29T20:00:00,None,None,[],False,NaN
3,cff46791-5ca5-11f0-b4be-d5cc39e1dd13,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,TBV Lemgo Lippe vs. SC Magdeburg,...,None,True,2025-12-31T04:31:10,2025-07-09T09:19:27,2025-08-29T21:00:00,None,None,[],False,NaN
4,d236ce04-5ca5-11f0-b4f5-d5cc39e1dd13,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,ThSV Eisenach vs. SC DHfK Leipzig,...,None,True,2025-12-31T04:31:10,2025-07-09T09:19:31,2025-08-30T17:00:00,None,None,[],False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,6dbe5959-5ca6-11f0-abf0-2d244033710e,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,VfL Gummersbach vs. Handball Sport Verein Hamburg,...,None,True,2025-12-31T04:33:20,2025-07-09T09:23:52,2025-12-27T20:00:00,None,None,[],False,NaN
168,70008f3d-5ca6-11f0-bae9-6f751d203d7c,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,ThSV Eisenach vs. SC Magdeburg,...,None,True,2025-12-31T04:33:24,2025-07-09T09:23:56,2025-12-27T20:00:00,None,None,[],False,NaN
169,6b759efa-5ca6-11f0-a1f0-0716b158c59d,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,Füchse Berlin vs. FRISCH AUF! Göppingen,...,None,True,2025-12-31T04:33:21,2025-07-09T09:23:48,2025-12-27T21:00:00,None,None,[],False,NaN
170,6f069ec1-5ca6-11f0-86fd-83b53cde89a1,h1s44,"{'resourceType': 'organizations', 'id': 'h1s44'}",b23932f0-5ca5-11f0-a95f-55240abd96d4,"{'resourceType': 'seasons', 'id': 'b23932f0-5c...",None,None,CONFIRMED,None,SC DHfK Leipzig vs. Rhein-Neckar Löwen,...,None,True,2025-12-31T04:33:24,2025-07-09T09:23:54,2025-12-27T21:00:00,None,None,[],False,NaN


In [7]:
# kinexon teams for the season
teams_kinexon = kinexon_get_teams_for_season(api_kinexon, str(YEAR))
print(f"Kinexon teams for season {YEAR}:")
print(teams_kinexon)

Kinexon teams for season 2025:
    id                    name
0   13           Füchse Berlin
1   16            SC Magdeburg
2    6            MT Melsungen
3   18                THW Kiel
4    5         TBV Lemgo Lippe
5    8   TSV Hannover-Burgdorf
6    7      Rhein-Neckar Löwen
7    9         SC DHFK Leipzig
8    3             HSG Wetzlar
9   23             HSV Hamburg
10  33           ThSV Eisenach
11  12  SG Flensburg-Handewitt
12  11   Frisch Auf! Göppingen
13   2             HC Erlangen
14  17           TVB Stuttgart
15  32         VfL Gummersbach
16  14           Bergischer HC
17  19              GWD Minden


In [8]:
start_time_min = pd.to_datetime(fixtures["startTimeLocal"]).min()
end_time_max = pd.to_datetime(fixtures["startTimeLocal"]).max()
df_team_sessions = kinexon_get_sessions_for_team(
    api=api_kinexon,
    team_id=0,
    start_date=start_time_min,
    end_date=end_time_max,
)
display(df_team_sessions)



,id,facility,types,group_behaviour,track_gps_always,only_imu_data,team_id,team_name,track_outside_field,track_while_clock_stopped,...,start_session,end_session,duration,type,description,timezone_id,group_assignment,group_colors,group_names,phases
0,3024,"{'id': 10, 'name': '281119', 'type': 'handball...","[{'label': 'Match', 'sortOrder': None, 'childr...","[offline, team, team, ball, officials]",False,False,None,,1,False,...,2025-08-28T16:49:09,2025-08-28T18:54:33,7524,Match,HC Erlangen vs. THW Kiel,385,"{""1"":[{""player_id"":5,""function"":""LA""},{""player...","[black, #565856, #ffffff, #FCFF0D, #00E852]","[Offline, HC Erlangen, THW Kiel, Ball, Referee]","[{'facility': {'id': 10, 'name': '281119', 'ty..."
1,3025,"{'id': 40, 'name': 'Rothenbach Halle', 'type':...","[{'label': 'Match', 'sortOrder': None, 'childr...","[offline, team, team, ball, officials]",False,False,None,,1,False,...,2025-08-29T16:50:13,2025-08-29T18:56:35,7582,Match,MT Melsungen vs. Rhein-Neckar Löwen,385,"{""1"":[{""player_id"":1465,""function"":""RA""},{""pla...","[black, #ff0000, #54A9D5, #FF69E3, #BCE8A8]","[Offline, MT Melsungen, Rhein-Neckar Löwen, Ba...","[{'facility': {'id': 40, 'name': 'Rothenbach H..."
2,3026,"{'id': 8, 'name': 'NEU TUI Arena Presse', 'typ...","[{'label': 'Match', 'sortOrder': None, 'childr...","[offline, team, team, ball, officials]",False,False,None,,1,False,...,2025-08-29T17:53:30,2025-08-29T19:36:38,6188,Match,TBV Lemgo Lippe vs. SC Magdeburg,385,"{""1"":[{""player_id"":65,""function"":""RM""},{""playe...","[black, #54A9D5, #ecf3ef, #feeb72, #BCE8A8]","[Offline, TBV Lemgo Lippe, SC Magdeburg, Ball,...","[{'facility': {'id': 8, 'name': 'NEU TUI Arena..."
3,3027,"{'id': 16, 'name': 'Flens-Arena', 'type': 'han...","[{'label': 'Match', 'sortOrder': None, 'childr...","[offline, team, team, ball, team]",False,False,None,,1,False,...,2025-08-30T13:53:23,2025-08-30T15:43:59,6636,Match,ThSV Eisenach vs. SC DHfK Leipzig,385,"{""1"":[{""player_id"":1776,""function"":""RR""},{""pla...","[black, #54A9D5, #FF6347, #00E852, #BCE8A8]","[Offline, ThSV Eisenach, SC DHFK Leipzig, Ball...","[{'facility': {'id': 16, 'name': 'Flens-Arena'..."
4,3037,"{'id': 16, 'name': 'Flens-Arena', 'type': 'han...","[{'label': 'Match', 'sortOrder': None, 'childr...","[offline, team, team, ball, officials]",False,False,None,,1,False,...,2025-08-30T16:51:52,2025-08-30T18:44:40,6768,Match,HSG Wetzlar vs. SG Flensburg-Handewitt,385,"{""1"":[{""player_id"":2263,""function"":""KR""},{""pla...","[black, #025a03, #ededed, #54A9D5, #BCE8A8]","[Offline, HSG Wetzlar, SG Flensburg-Handewitt,...","[{'facility': {'id': 16, 'name': 'Flens-Arena'..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,3228,"{'id': 16, 'name': 'Flens-Arena', 'type': 'han...","[{'label': 'Match', 'sortOrder': None, 'childr...","[offline, team, team, ball, officials]",False,False,None,,1,False,...,2025-12-27T16:51:49,2025-12-27T18:40:21,6512,Match,ThSV Eisenach vs. SC Magdeburg,385,"{""1"":[{""player_id"":1776,""function"":""RR""},{""pla...","[black, #54A9D5, #00E852, #FF6347, #00E852]","[Offline, ThSV Eisenach, SC Magdeburg, Ball, S...","[{'facility': {'id': 16, 'name': 'Flens-Arena'..."
167,3229,"{'id': 16, 'name': 'Flens-Arena', 'type': 'han...","[{'label': 'Match', 'sortOrder': None, 'childr...","[offline, team, team, ball, officials]",False,False,None,,1,False,...,2025-12-27T16:52:24,2025-12-27T18:45:26,6782,Match,VfL Gummersbach vs. HSV Hamburg,385,"{""1"":[{""player_id"":1963,""function"":""TW""},{""pla...","[black, #54A9D5, #FF6347, #C5FFFA, #BCE8A8]","[Offline, VfL Gummersbach, HSV Hamburg, Ball, ...","[{'facility': {'id': 16, 'name': 'Flens-Arena'..."
168,3230,"{'id': 16, 'name': 'Flens-Arena', 'type': 'han...","[{'label': 'Match', 'sortOrder': None, 'childr...","[offline, team, team, ball, officials]",False,False,None,,1,False,...,2025-12-27T17:51:31,2025-12-27T19:39:27,6476,Match,Füchse Berlin vs. Frisch Auf! Göppingen,385,"{""1"":[{""player_id""

In [9]:
df_normalized_matches = normalize_matches(
    df_fixtures_sportradar_raw=fixtures,
    df_sessions_kinexon_raw=df_team_sessions,
    # df_teams_sportradar_raw=teams_sportradar_raw,
    # df_teams_kinexon_raw=teams_kinexon_raw,
)
display(df_normalized_matches)

# display only rows with nan in them
print("Not matched fixtures details:")
df_matched_details = df_normalized_matches[df_normalized_matches.isna().any(axis=1)]
display(df_matched_details)

,season_id,fixture_id,session_id,start_time_local,start_time_utc,start_time,round_number,team_name_home,team_name_away,entity_id_home,entity_id_away,score_home,score_away,start_session,description,match_name,attendance,team_id_home_kinexon,team_id_away_kinexon
143,b23932f0-5ca5-11f0-a95f-55240abd96d4,ccb6fb61-5ca5-11f0-b602-991bd49151a6,None,2025-08-27T19:00:00,2025-08-27T17:00:00,None,1,TSV Hannover-Burgdorf,VfL Gummersbach,fea93a14-3952-11ef-a7e0-af5c55c3771d,fe911367-3952-11ef-9131-af5c55c3771d,26,29,None,None,TSV Hannover-Burgdorf vs. VfL Gummersbach,9455.0,NaN,NaN
35,b23932f0-5ca5-11f0-a95f-55240abd96d4,d7e307a9-5ca5-11f0-af4b-018ba42b97a2,3024.0,2025-08-28T19:00:00,2025-08-28T17:00:00,2025-08-28T17:00:00+0000,1,HC Erlangen,THW Kiel,fe8d1885-3952-11ef-9130-af5c55c3771d,febf038e-3952-11ef-b7c2-af5c55c3771d,29,31,2025-08-28T16:49:09,HC Erlangen vs. THW Kiel,HC Erlangen vs. THW Kiel,6924.0,2.0,18.0
60,b23932f0-5ca5-11f0-a95f-55240abd96d4,c9ebe4b9-5ca5-11f0-b9d0-d509415b4f76,3025.0,2025-08-29T19:00:00,2025-08-29T17:00:00,2025-08-29T17:00:00+0000,1,MT Melsungen,Rhein-Neckar Löwen,feb41b46-3952-11ef-9df6-af5c55c3771d,fe80598a-3952-11ef-914c-af5c55c3771d,27,29,2025-08-29T16:50:13,MT Melsungen vs. Rhein-Neckar Löwen,MT Melsungen vs. Rhein-Neckar Löwen,3979.0,6.0,7.0
107,b23932f0-5ca5-11f0-a95f-55240abd96d4,cff46791-5ca5-11f0-b4be-d5cc39e1dd13,3026.0,2025-08-29T20:00:00,2025-08-29T18:00:00,2025-08-29T18:00:00+0000,1,TBV Lemgo Lippe,SC Magdeburg,fe99a935-3952-11ef-9dd4-af5c55c3771d,fe848316-3952-11ef-8185-af5c55c3771d,29,33,2025-08-29T17:53:30,TBV Lemgo Lippe vs. SC Magdeburg,TBV Lemgo Lippe vs. SC Magdeburg,4308.0,5.0,16.0
156,b23932f0-5ca5-11f0-a95f-55240abd96d4,d236ce04-5ca5-11f0-b4f5-d5cc39e1dd13,3027.0,2025-08-30T16:00:00,2025-08-30T14:00:00,2025-08-30T14:00:00+0000,1,ThSV Eisenach,SC DHfK Leipzig,fef51771-3952-11ef-97b4-af5c55c3771d,feace61d-3952-11ef-ae23-af5c55c3771d,31,27,2025-08-30T13:53:23,ThSV Eisenach vs. SC DHfK Leipzig,ThSV Eisenach vs. SC DHfK Leipzig,2750.0,33.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,b23932f0-5ca5-11f0-a95f-55240abd96d4,05fd729b-5ca6-11f0-8efe-a7e47f7b55aa,3126.0,2025-10-18T20:00:00,2025-10-18T18:00:00,2025-10-18T18:00:00+0000,9,THW Kiel,MT Melsungen,febf038e-3952-11ef-b7c2-af5c55c3771d,feb41b46-3952-11ef-9df6-af5c55c3771d,31,29,2025-10-18T17:53:20,THW Kiel vs. MT Melsungen,THW Kiel vs. MT Melsungen,10285.0,18.0,6.0
146,b23932f0-5ca5-11f0-a95f-55240abd96d4,0b248d78-5ca6-11f0-9c92-39512e9bd225,3127.0,2025-10-19T15:00:00,2025-10-19T13:00:00,2025-10-19T13:00:00+0000,9,TVB Stuttgart,Füchse Berlin,febb3114-3952-11ef-b6f7-af5c55c3771d,fe7bdd16-3952-11ef-b585-af5c55c3771d,30,36,2025-10-19T12:49:55,TVB Stuttgart vs. Füchse Berlin,TVB Stuttgart vs. Füchse Berlin,5540.0,17.0,13.0
79,b23932f0-5ca5-11f0-a95f-55240abd96d4,0788ecda-5ca6-11f0-baf1-a7e47f7b55aa,3128.0,2025-10-19T16:00:00,2025-10-19T14:00:00,2025-10-19T14:00:00+0000,9,SC DHfK Leipzig,SC Magdeburg,feace61d-3952-11ef-ae23-af5c55c3771d,fe848316-3952-11ef-8185-af5c55c3771d,23,36,2025-10-19T13:53:14,SC DHfK Leipzig vs. SC Magdeburg,SC DHfK Leipzig vs. SC Magdeburg,6000.0,9.0,16.0
55,b23932f0-5ca5-11f0-a95f-55240abd96d4,090fadb2-5ca6-11f0-bb5d-11b226ba5eb7,3131.0,2025-10-19T16:30:00,2025-10-19T14:30:00,2025-10-19T14:30:00+0000,9,HSV Hamburg,VfL Gummersbach,0045fbc4-3953-11ef-a217-af5c55c3771d,fe911367-3952-11ef-9131-af5c55c3771d,31,30,2025-10-19T14:20:47,HSV Hamburg vs. VfL Gummersbach,Handball Sport Verein Hamburg vs. VfL Gummersbach,4870.0,23.0,32.0


Not matched fixtures details:


,season_id,fixture_id,session_id,start_time_local,start_time_utc,start_time,round_number,team_name_home,team_name_away,entity_id_home,entity_id_away,score_home,score_away,start_session,description,match_name,attendance,team_id_home_kinexon,team_id_away_kinexon
143,b23932f0-5ca5-11f0-a95f-55240abd96d4,ccb6fb61-5ca5-11f0-b602-991bd49151a6,None,2025-08-27T19:00:00,2025-08-27T17:00:00,None,1,TSV Hannover-Burgdorf,VfL Gummersbach,fea93a14-3952-11ef-a7e0-af5c55c3771d,fe911367-3952-11ef-9131-af5c55c3771d,26,29,None,None,TSV Hannover-Burgdorf vs. VfL Gummersbach,9455.0,NaN,NaN


In [10]:
# # fuzzy match sessions to fixtures by using "description" and "nameLocal" and the date
# from thefuzz import process

# def match_sessions_to_fixtures(sessions: pd.DataFrame, fixtures: pd.DataFrame) -> pd.DataFrame:
#     fixture_names = fixtures["nameLocal"].tolist()
#     matched_fixtures = []
#     for _, session in sessions.iterrows():
#         session_desc = session["description"]
#         if pd.isna(session_desc):
#             continue
#         best_match, score = process.extractOne(session_desc, fixture_names)
#         if score >= 80:  # threshold for a good match
#             matched_fixture = fixtures[fixtures["nameLocal"] == best_match].iloc[0]
#             matched_fixtures.append({
#                 "session_id": session["id"],
#                 "fixture_id": matched_fixture["fixtureId"],
#                 "match_score": score
#             })
#     return pd.DataFrame(matched_fixtures)

# df_matched_sessions_fixtures = match_sessions_to_fixtures(df_team_sessions, fixtures)

# # stats how many sessions were matched
# total_sessions = len(df_team_sessions)
# matched_sessions = len(df_matched_sessions_fixtures)
# print(f"Total sessions: {total_sessions}, Matched sessions: {matched_sessions}, Match rate: {matched_sessions/total_sessions:.2%}")
# # unmatched sessions
# unmatched_sessions = df_team_sessions[~df_team_sessions["id"].isin(df_matched_sessions_fixtures["session_id"])]
# print("Unmatched sessions:")
# display(unmatched_sessions)
# # display matched sessions with fixture details
# df_matched_details = df_matched_sessions_fixtures.merge(
#     fixtures,
#     left_on="fixture_id",
#     right_on="fixtureId",
#     how="left"
# )
# print("Matched sessions with fixture details:")
# display(df_matched_details)

In [ ]:
for _, fixture in df_normalized_matches.iterrows():
    fixture_id = fixture["fixture_id"]
    fixture_gameday = fixture["round_number"]
    fixture_name = fixture["description"]
    fixture_start_time = fixture["start_time_local"]
    session_id = int(fixture["session_id"]) if not pd.isna(fixture["session_id"]) else None
    print(
        f"Gameday {fixture_gameday} | Time {fixture_start_time} | Name {fixture_name} | SessionId {session_id} | Processing fixture ID: {fixture_id}"
    )
    if fixture_name is None:
        continue
    # check if flensburg is home, else skip
    if "Flensburg" not in fixture_name.split("vs")[0]:
        print("Skipping fixture as Flensburg is not home.")
        continue

    # Fetch fixture events
    fixture_events = sr_get_fixture_events(api_sportradar, fixture_id)
    # get players
    players = sr_get_players_for_fixture(api_sportradar, fixture_events)

    # fetch kinexon events
    detected_events = kinexon_get_detected_events_for_fixture(
        api_kinexon, session_id
    )
    path_file_positions = os.path.join(
        POSITIONS_CACHE_PATH, f"kinexon_positions_{session_id}.0.parquet.gzip"
    )

    # check if file already exist
    if os.path.exists(path_file_positions):
        print(
            f"File kinexon_positions_{session_id}.0.parquet.gzip already exists, loading."
        )
        positions = pd.read_parquet(path_file_positions)
    else:
        # remove once tested
        continue
        # fetch kinexon positions
        positions = kinexon_get_positions_for_session(api_kinexon, session_id)
        # for positions insert session_id
        positions["session_id"] = session_id
        positions["fixture_id"] = fixture_id
        # save as kinexon_positions_{'session_id'}.0.parquet.gzip
        positions.to_parquet(
            f"kinexon_positions_{session_id}.0.parquet.gzip",
            compression="gzip",
        )

    detected_events = kinexon_get_detected_events_for_fixture(
        api_kinexon, session_id
    )
    # insert fixture_id
    detected_events["fixture_id"] = fixture_id
    positions["session_id"] = session_id
    positions["fixture_id"] = fixture_id
    fixture_events["fixture_id"] = fixture_id
    players["fixture_id"] = fixture_id

    # display(detected_events)
    # # Normalize data
    normalized_detected_events = normalize_match_detected_shots(
        detected_events
    )
    normalized_positions = normalize_match_positions(positions)
    normalized_events = normalize_match_events(fixture_events)
    normalized_setup = normalize_match_events_setup(normalized_events)
    normalized_goals = normalize_match_events_goals(normalized_events)
    normalized_players = normalize_match_players(players)

    # funny hack for flensburg. if flensburg is in the team home, there is a y offset of 12.5 m
    if fixture["team_name_home"] == "SG Flensburg-Handewitt":
        print("Applying Flensburg Y offset hack.")
        normalized_positions.loc[:, "y_m"] = normalized_positions["y_m"] - 12.5

    # sync data
    synced_players = extract_players_for_match(
        pd.DataFrame(fixture).T,
        normalized_setup,
        normalized_detected_events,
        normalized_positions,
        normalized_players,
    )
    print("Synced players:")
    display(synced_players)

    synced_shot_events = sync_shot_events(
        df_match_normalized=pd.DataFrame(fixture).T,
        df_match_events_normalized_goals=normalized_goals,
        df_match_detected_shots_normalized=normalized_detected_events,
        df_positions_normalized=normalized_positions,
        df_players=synced_players,
    )
    print(f"Synced shot events. Total goal events: {len(normalized_goals)}")
    display(synced_shot_events)

    rendered = render_shot_event(
            df_goals=synced_shot_events,
            df_positions=normalized_positions,
            max_events=5,
            require_throw_ts=False,
            output_dir="../../output/shot_event_renders/",
        )

    df_xg_features = calculate_xg_features(
        df_match_normalized=pd.DataFrame(fixture).T,
        df_shot_events=synced_shot_events,
        df_positions_normalized=normalized_positions,
    )
    print("XG Features:")
    display(df_xg_features)
    # break

Gameday 1 | Time 2025-08-27T19:00:00 | Name None | SessionId None | Processing fixture ID: ccb6fb61-5ca5-11f0-b602-991bd49151a6
Gameday 1 | Time 2025-08-28T19:00:00 | Name HC Erlangen vs. THW Kiel | SessionId 3024 | Processing fixture ID: d7e307a9-5ca5-11f0-af4b-018ba42b97a2
Skipping fixture as Flensburg is not home.
Gameday 1 | Time 2025-08-29T19:00:00 | Name MT Melsungen vs. Rhein-Neckar Löwen | SessionId 3025 | Processing fixture ID: c9ebe4b9-5ca5-11f0-b9d0-d509415b4f76
Skipping fixture as Flensburg is not home.
Gameday 1 | Time 2025-08-29T20:00:00 | Name TBV Lemgo Lippe vs. SC Magdeburg | SessionId 3026 | Processing fixture ID: cff46791-5ca5-11f0-b4be-d5cc39e1dd13
Skipping fixture as Flensburg is not home.
Gameday 1 | Time 2025-08-30T16:00:00 | Name ThSV Eisenach vs. SC DHfK Leipzig | SessionId 3027 | Processing fixture ID: d236ce04-5ca5-11f0-b4f5-d5cc39e1dd13
Skipping fixture as Flensburg is not home.
Gameday 1 | Time 2025-08-30T19:00:00 | Name HSG Wetzlar vs. SG Flensburg-Handewi